# 🎬 MovieLens 32M — Data Preprocessing Pipeline

This notebook handles the complete preprocessing of the MovieLens 32M dataset:
- Data loading & inspection
- Missing values & duplicate handling
- Data type optimization (memory reduction)
- Timestamp → datetime conversion
- Feature engineering (genres, release year)
- Tag normalization
- User & movie aggregate statistics
- Sparse data filtering
- Export cleaned data

## 1. Imports & Configuration

In [7]:
import os
import time
import pandas as pd
import numpy as np
from datetime import datetime

# Configuration
DATA_DIR = "ml-32m"
OUT_DIR = "processed"
os.makedirs(OUT_DIR, exist_ok=True)

# Display settings
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 20)
pd.set_option("display.float_format", "{:.2f}".format)

print("✅ Imports loaded")

✅ Imports loaded


## 2. Load Raw Data

In [9]:
t0 = time.time()

movies  = pd.read_csv(f"{DATA_DIR}/movies.csv")
links   = pd.read_csv(f"{DATA_DIR}/links.csv")
tags    = pd.read_csv(f"{DATA_DIR}/tags.csv")
ratings = pd.read_csv(f"{DATA_DIR}/ratings.csv")

print(f"⏱  Loaded in {time.time()-t0:.1f}s")
print(f"   movies  : {movies.shape}")
print(f"   links   : {links.shape}")
print(f"   tags    : {tags.shape}")
print(f"   ratings : {ratings.shape}")

⏱  Loaded in 6.4s
   movies  : (87585, 3)
   links   : (87585, 3)
   tags    : (2000072, 4)
   ratings : (32000204, 4)


## 3. Data Quality Inspection

In [11]:
# Quick look at each dataframe
print("=" * 50)
print("MOVIES")
print("=" * 50)
display(movies.head())
print(f"\nShape: {movies.shape}")
print(f"Dtypes:\n{movies.dtypes}")
print(f"\nMissing values:\n{movies.isnull().sum()}")
print(f"Duplicates: {movies.duplicated().sum()}")

MOVIES


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy



Shape: (87585, 3)
Dtypes:
movieId    int64
title        str
genres       str
dtype: object

Missing values:
movieId    0
title      0
genres     0
dtype: int64
Duplicates: 0


In [12]:
print("=" * 50)
print("LINKS")
print("=" * 50)
display(links.head())
print(f"\nShape: {links.shape}")
print(f"Dtypes:\n{links.dtypes}")
print(f"\nMissing values:\n{links.isnull().sum()}")
print(f"Duplicates: {links.duplicated().sum()}")

LINKS


,movieId,imdbId,tmdbId
0,1,114709,862.00
1,2,113497,8844.00
2,3,113228,15602.00
3,4,114885,31357.00
4,5,113041,11862.00



Shape: (87585, 3)
Dtypes:
movieId      int64
imdbId       int64
tmdbId     float64
dtype: object

Missing values:
movieId      0
imdbId       0
tmdbId     124
dtype: int64
Duplicates: 0


## 4. Handle Missing Values & Duplicates

In [13]:
# --- Links: fill missing tmdbId with -1 ---
if links["tmdbId"].dtype == "float64":
    n_missing_tmdb = links["tmdbId"].isnull().sum()
    links["tmdbId"] = links["tmdbId"].fillna(-1).astype(int)
    print(f"links.tmdbId: filled {n_missing_tmdb} missing with -1")
else:
    print("links.tmdbId: already clean")

# --- Remove duplicate ratings ---
before = len(ratings)
ratings = ratings.drop_duplicates()
print(f"ratings: removed {before - len(ratings)} duplicates")

print("Done")

links.tmdbId: filled 124 missing with -1
ratings: removed 0 duplicates
Done


## 5. Optimise Data Types (Memory Reduction)

In [14]:
def mem_mb(df):
    return df.memory_usage(deep=True).sum() / 1024**2

# --- Ratings ---
before_mem = mem_mb(ratings)
ratings["userId"]  = ratings["userId"].astype("int32")
ratings["movieId"] = ratings["movieId"].astype("int32")
ratings["rating"]  = ratings["rating"].astype("float32")
after_mem = mem_mb(ratings)
print(f"ratings: {before_mem:.0f} MB -> {after_mem:.0f} MB (saved {before_mem - after_mem:.0f} MB)")

# --- Movies & Links ---
movies["movieId"] = movies["movieId"].astype("int32")
links["movieId"]  = links["movieId"].astype("int32")
links["imdbId"]   = links["imdbId"].astype("int32")
links["tmdbId"]   = links["tmdbId"].astype("int32")

print("Data types optimised (tags handled in Step 8)")

ratings: 977 MB -> 610 MB (saved 366 MB)
Data types optimised (tags handled in Step 8)


## 6. Convert Timestamps to Datetime

In [15]:
# --- Convert timestamps to datetime ---
if "timestamp" in ratings.columns:
    ratings["datetime"] = pd.to_datetime(ratings["timestamp"], unit="s")
    ratings["year"]  = ratings["datetime"].dt.year.astype("int16")
    ratings["month"] = ratings["datetime"].dt.month.astype("int8")
    ratings.drop(columns=["timestamp"], inplace=True)
    print(f"Ratings date range: {ratings["datetime"].min()} -> {ratings["datetime"].max()}")
else:
    print("Timestamps already converted")

display(ratings.head())

Ratings date range: 1995-01-09 11:46:44 -> 2023-10-13 02:29:07


,userId,movieId,rating,datetime,year,month
0,1,17,4.00,1999-12-03 19:24:37,1999,12
1,1,25,1.00,1999-12-03 19:43:48,1999,12
2,1,29,2.00,1999-11-22 00:36:16,1999,11
3,1,30,5.00,1999-12-03 19:24:37,1999,12
4,1,32,5.00,1999-11-22 00:00:58,1999,11


## 7. Feature Engineering — Movies

In [16]:
# --- Extract release year from title ---
movies["release_year"] = (
    movies["title"]
    .str.extract(r"\((\d{4})\)$")
    .astype("float")
    .astype("Int32")
)
n_no_year = movies["release_year"].isna().sum()
print(f"Extracted release_year ({n_no_year} movies had no parseable year)")

# --- Convert genres from pipe-separated string to list ---
def parse_genres(x):
    if isinstance(x, list):
        return x  # already converted
    if not isinstance(x, str) or x == "(no genres listed)":
        return []
    return [g.strip() for g in x.split("|") if g.strip() != "(no genres listed)"]

movies["genres"] = movies["genres"].apply(parse_genres)
print(f"Converted genres to list format")
print(f"Movies with no genres: {movies["genres"].apply(len).eq(0).sum()}")
display(movies[["movieId", "title", "genres", "release_year"]].head())

Extracted release_year (771 movies had no parseable year)
Converted genres to list format
Movies with no genres: 7080


,movieId,title,genres,release_year
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",1995
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]",1995
2,3,Grumpier Old Men (1995),"[Comedy, Romance]",1995
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]",1995
4,5,Father of the Bride Part II (1995),[Comedy],1995


## 8. Normalise Tags

In [17]:
# Normalise and aggregate tags per movie
tags_raw = pd.read_csv(f"{DATA_DIR}/tags.csv")
tags_raw = tags_raw.dropna(subset=["tag"])
tags_raw["tag_clean"] = tags_raw["tag"].str.strip().str.lower()

n_unique_raw   = tags_raw["tag"].nunique()
n_unique_clean = tags_raw["tag_clean"].nunique()
print(f"Unique tags: {n_unique_raw:,} -> {n_unique_clean:,} (reduced {n_unique_raw - n_unique_clean:,})")

# Aggregate into flat list per movieId
tags = tags_raw.groupby("movieId")["tag_clean"].apply(list).reset_index()
tags.columns = ["movieId", "tag"]
print(f"Aggregated tags for {len(tags):,} movies")
display(tags.tail())

Unique tags: 140,979 -> 131,645 (reduced 9,334)
Aggregated tags for 51,323 movies


,movieId,tag
51318,292143,"[cadaqués, catalonia, china, housing estate, h..."
51319,292349,[politically incorrect]
51320,292371,[stephen king]
51321,292597,[artificial intelligence]
51322,292629,"[documentary filmmaking, family relationships]"


## 9. Merge All Data

In [18]:
# Merge ratings with movies (title + genres)
df = ratings.merge(movies[["movieId", "title", "genres"]], on="movieId", how="left")

# Merge with grouped tags (flat list per movie)
df = df.merge(tags[["movieId", "tag"]], on="movieId", how="left")

# Drop datetime, year, month if present
df = df.drop(columns=["datetime", "year", "month"], errors="ignore")

# Fill missing tags with empty list
df["tag"] = df["tag"].apply(lambda x: x if isinstance(x, list) else [])

# Re-encode userId as sequential integers starting from 1
user_map = {uid: idx + 1 for idx, uid in enumerate(sorted(df["userId"].unique()))}
df["userId"] = df["userId"].map(user_map)

# Reset index starting from 1
df = df.reset_index(drop=True)
df.index = df.index + 1
df.index.name = None

print(f"userId range: {df["userId"].min()} to {df["userId"].max()}")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

# Show first rating from 10 different users
display(df.groupby("userId").first().head(10).reset_index())
display(df.sample(5, random_state=42))

userId range: 1 to 200948
Shape: (32000204, 6)
Columns: ['userId', 'movieId', 'rating', 'title', 'genres', 'tag']


,userId,movieId,rating,title,genres,tag
0,1,17,4.00,Sense and Sensibility (1995),"[Drama, Romance]","[boring, nothing happens, 18th century, classi..."
1,2,31,5.00,Dangerous Minds (1995),[Drama],"[teacher changing lives, based on a book, insp..."
2,3,2,3.50,Jumanji (1995),"[Adventure, Children, Fantasy]","[robin williams, fantasy, robin williams, time..."
3,4,223,4.00,Clerks (1994),[Comedy],"[crude humor, cynical, funny, good dialogue, g..."
4,5,10,4.00,GoldenEye (1995),"[Action, Adventure, Thriller]","[james bond, tumey's dvds, 007, bond, casual s..."
5,6,590,4.00,Dances with Wolves (1990),"[Adventure, Drama, Western]","[wolves, can see 100 times no bore at all, osc..."
6,7,19,3.00,Ace Ventura: When Nature Calls (1995),[Comedy],"[jim carrey, detective, silly fun, very funny,..."
7,8,32,4.00,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),"[Mystery, Sci-Fi, Thriller]","[atmospheric, brad pitt, bruce willis, design,..."
8,9,32,4.00,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),"[Mystery, Sci-Fi, Thriller]","[atmospheric, brad pitt, bruce willis, design,..."
9,10,1,2.50,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]","[children, disney, animation, children, disney..."


,userId,movieId,rating,title,genres,tag
10685862,66954,781,5.00,Stealing Beauty (1996),[Drama],"[jeremy irons, emotion!, atmospheric, erotic, ..."
1552724,9877,574,4.00,Spanking the Monkey (1994),"[Comedy, Drama]","[directorial debut, incest, nudity (full front..."
6145185,38348,1088,2.00,Dirty Dancing (1987),"[Drama, Musical, Romance]","[80's classic, coming of age, dance, dancing, ..."
16268585,101952,2706,1.00,American Pie (1999),"[Comedy, Romance]","[teenager, american pie series, comedy, high s..."
22418635,140400,275079,3.50,Chip 'n Dale: Rescue Rangers (2022),"[Adventure, Animation, Children, Comedy, Fanta...","[creative plot, meta, original plot, parody, u..."


## 9. Compute Aggregate Statistics

In [19]:
# --- Per-User Stats ---
user_stats = ratings.groupby("userId").agg(
    n_ratings   = ("rating", "size"),
    mean_rating = ("rating", "mean"),
    std_rating  = ("rating", "std"),
    min_rating  = ("rating", "min"),
    max_rating  = ("rating", "max"),
).reset_index()
user_stats["std_rating"]  = user_stats["std_rating"].fillna(0).astype("float32")
user_stats["mean_rating"] = user_stats["mean_rating"].astype("float32")

print(f"👤 user_stats shape: {user_stats.shape}")
display(user_stats.describe())

👤 user_stats shape: (200948, 6)


,userId,n_ratings,mean_rating,std_rating,min_rating,max_rating
count,200948.00,200948.00,200948.00,200948.00,200948.00,200948.00
mean,100474.50,159.25,3.70,0.92,1.32,4.97
std,58008.84,282.03,0.49,0.27,0.85,0.15
min,1.00,20.00,0.50,0.00,0.50,0.50
25%,50237.75,36.00,3.42,0.73,0.50,5.00
50%,100474.50,73.00,3.72,0.89,1.00,5.00
75%,150711.25,167.00,4.03,1.08,2.00,5.00
max,200948.00,33332.00,5.00,2.31,5.00,5.00


In [20]:
# --- Per-Movie Stats ---
movie_stats = ratings.groupby("movieId").agg(
    n_ratings   = ("rating", "size"),
    mean_rating = ("rating", "mean"),
    std_rating  = ("rating", "std"),
    min_rating  = ("rating", "min"),
    max_rating  = ("rating", "max"),
).reset_index()
movie_stats["std_rating"]  = movie_stats["std_rating"].fillna(0).astype("float32")
movie_stats["mean_rating"] = movie_stats["mean_rating"].astype("float32")

print(f"🎬 movie_stats shape: {movie_stats.shape}")
display(movie_stats.describe())

🎬 movie_stats shape: (84432, 6)


,movieId,n_ratings,mean_rating,std_rating,min_rating,max_rating
count,84432.00,84432.00,84432.00,84432.00,84432.00,84432.00
mean,157208.73,379.01,3.01,0.75,1.69,4.02
std,79959.54,2592.44,0.80,0.55,1.24,1.11
min,1.00,1.00,0.50,0.00,0.50,0.50
25%,109373.50,2.00,2.54,0.26,0.50,3.50
50%,166596.00,5.00,3.07,0.86,1.50,4.50
75%,213535.50,25.00,3.50,1.09,2.50,5.00
max,292757.00,102929.00,5.00,3.18,5.00,5.00


## 10. Filter Sparse Users & Movies

In [21]:
MIN_USER_RATINGS  = 20   # already guaranteed by dataset
MIN_MOVIE_RATINGS = 2    # remove very obscure movies

active_movies = movie_stats[movie_stats["n_ratings"] >= MIN_MOVIE_RATINGS]["movieId"]
ratings_filtered = ratings[ratings["movieId"].isin(active_movies)]

removed = len(ratings) - len(ratings_filtered)
print(f"🔍 Removed {removed:,} ratings for movies with < {MIN_MOVIE_RATINGS} ratings")
print(f"   Movies : {ratings['movieId'].nunique():,} → {ratings_filtered['movieId'].nunique():,}")
print(f"   Ratings: {len(ratings):,} → {len(ratings_filtered):,}")

🔍 Removed 18,607 ratings for movies with < 2 ratings
   Movies : 84,432 → 65,825
   Ratings: 32,000,204 → 31,981,597


## 11. Save Processed Data

In [23]:
movies.to_csv(f"{OUT_DIR}/movies_processed.csv", index=False)
links.to_csv(f"{OUT_DIR}/links_processed.csv", index=False)
tags.to_csv(f"{OUT_DIR}/tags_processed.csv", index=False)
user_stats.to_csv(f"{OUT_DIR}/user_stats.csv", index=False)
movie_stats.to_csv(f"{OUT_DIR}/movie_stats.csv", index=False)
ratings_filtered.to_csv(f"{OUT_DIR}/ratings_filtered.csv", index=False)

print(f"\n💾 All files saved to ./{OUT_DIR}/")
for f in sorted(os.listdir(OUT_DIR)):
    size = os.path.getsize(f"{OUT_DIR}/{f}") / 1024**2
    print(f"   {f:30s} {size:8.1f} MB")


💾 All files saved to ./processed/
   links_processed.csv                 1.7 MB
   movie_stats.csv                     2.6 MB
   movies_processed.csv                4.8 MB
   ratings_filtered.csv             1307.7 MB
   tags_processed.csv                 29.3 MB
   user_stats.csv                      7.1 MB


## 12. Final Summary

In [24]:
# Calculate number of unique genres
n_genres = len(set(g for sublist in movies["genres"].dropna() if isinstance(sublist, list) for g in sublist))

print("=" * 55)
print("  PREPROCESSING COMPLETE")
print("=" * 55)
print(f"  Dataset        : MovieLens 32M")
print(f"  Users          : {ratings_filtered["userId"].nunique():,}")
print(f"  Movies         : {ratings_filtered["movieId"].nunique():,}")
print(f"  Ratings        : {len(ratings_filtered):,}")
print(f"  Tags           : {len(tags):,}")
print(f"  Genres encoded : {n_genres:,}")
print(f"  Date range     : {ratings_filtered["datetime"].min().date()} -> {ratings_filtered["datetime"].max().date()}")
print(f"  Output dir     : ./{OUT_DIR}/")
print("=" * 55)

  PREPROCESSING COMPLETE
  Dataset        : MovieLens 32M
  Users          : 200,948
  Movies         : 65,825
  Ratings        : 31,981,597
  Tags           : 51,323
  Genres encoded : 19
  Date range     : 1995-01-09 -> 2023-10-13
  Output dir     : ./processed/


# ---------------------------------------------------------
# 🎬 CONTENT-BASED RECOMMENDATION SYSTEM
# ---------------------------------------------------------

## 13. Recommender Preprocessing
Since we are in the same notebook, `movies` and `tags` are already loaded. We just need to merge them to create a single movie-level content dataframe (not the 32M rating dataframe).

In [25]:
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
import pickle
import time
import numpy as np

# 1. Merge movies with grouped tags
t0 = time.time()
movies_content = movies[["movieId", "title", "genres", "release_year"]].copy()
movies_content = movies_content.merge(tags[["movieId", "tag"]], on="movieId", how="left")

# Fill missing tags with empty list to prevent ast.eval or loop issues
movies_content["tag"] = movies_content["tag"].apply(lambda x: x if isinstance(x, list) else [])

# 2. Add title normalization for O(1) searches
movies_content["title_norm"] = movies_content["title"].str.lower().str.strip()

print(f"Recommender Dataset Shape: {movies_content.shape} (Time: {time.time()-t0:.2f}s)")
display(movies_content.head(3))

Recommender Dataset Shape: (87585, 6) (Time: 0.03s)


,movieId,title,genres,release_year,tag,title_norm
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",1995,"[children, disney, animation, children, disney...",toy story (1995)
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]",1995,"[robin williams, fantasy, robin williams, time...",jumanji (1995)
2,3,Grumpier Old Men (1995),"[Comedy, Romance]",1995,"[comedinha de velhinhos engraãƒâ§ada, comedinh...",grumpier old men (1995)


## 14. Advanced Feature Engineering (Weighted Content)
To obey the constraint of weighting **genres** higher than user **tags**, we will duplicate the genres string x3 before combining.

In [26]:
def build_weighted_content(row):
    # Extracting genres (repeating x3 for higher weighting)
    # If genres is NaN or float somehow, replace with empty
    g_list = row["genres"] if isinstance(row["genres"], list) else []
    genres_text = " ".join([g.lower().replace(" ", "") for g in g_list])
    weighted_genres = " ".join([genres_text] * 3)
    
    # Extracting tags as normal weighting
    t_list = row["tag"] if isinstance(row["tag"], list) else []
    tags_text = " ".join([t.lower().replace(" ", "") for t in t_list])
    
    # Combining with title
    return f"{str(row['title_norm'])} {weighted_genres} {tags_text}".strip()

t0 = time.time()
movies_content["content"] = movies_content.apply(build_weighted_content, axis=1)
print(f"Content generated in {time.time()-t0:.2f}s")
display(movies_content[["title", "content"]].head())

Content generated in 0.82s


,title,content
0,Toy Story (1995),toy story (1995) adventure animation children ...
1,Jumanji (1995),jumanji (1995) adventure children fantasy adve...
2,Grumpier Old Men (1995),grumpier old men (1995) comedy romance comedy ...
3,Waiting to Exhale (1995),waiting to exhale (1995) comedy drama romance ...
4,Father of the Bride Part II (1995),father of the bride part ii (1995) comedy come...


## 15. Vectorization (TF-IDF)
We use unigrams and bigrams, keeping vocabulary memory-controlled via `min_df` and `max_df`.

In [27]:
t0 = time.time()
tfidf = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.8,
    max_features=50000
)

tfidf_matrix = tfidf.fit_transform(movies_content["content"])
print(f"TF-IDF Matrix Vectorized in {time.time()-t0:.2f}s")
print(f"TF-IDF Matrix Shape: {tfidf_matrix.shape}")

TF-IDF Matrix Vectorized in 4.97s
TF-IDF Matrix Shape: (87585, 50000)


## 16. Fast O(1) Index Lookup
We map each `normalized_title` directly to its row index in the `tfidf_matrix` to avoid Pandas loop iteration on queries. We also map duplicates to gracefully ignore/overwrite.

In [28]:
title_to_idx = {}
for idx, row in movies_content.iterrows():
    t_norm = row["title_norm"]
    if t_norm not in title_to_idx:
        title_to_idx[t_norm] = idx
    # In event of a duplicate title, it will keep the first instance.

print(f"Created fast lookup dict with {len(title_to_idx):,} unique titles.")

Created fast lookup dict with 87,379 unique titles.


## 17. Recommender Persistence (Saving to Disk)
Instead of calculating this every restart, we serialize the elements so a web server could load them in seconds.

In [29]:
import os
# Ensure models output directory exists
MODEL_DIR = "models"
os.makedirs(f"{MODEL_DIR}", exist_ok=True)

# 1. Save Sparse TF-IDF Matrix (very small on disk)
sp.save_npz(f"{MODEL_DIR}/tfidf_matrix.npz", tfidf_matrix)

# 2. Save Model & Dictionary via Pickle
with open(f"{MODEL_DIR}/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)

with open(f"{MODEL_DIR}/title_to_idx.pkl", "wb") as f:
    pickle.dump(title_to_idx, f)

print("\n\U0001f4be Recommender assets saved successfully:")
for f in sorted(os.listdir(MODEL_DIR)):
    size = os.path.getsize(f"{MODEL_DIR}/{f}") / 1024**2
    print(f"   {f:30s} {size:8.1f} MB")


💾 Recommender assets saved successfully:
   tfidf_matrix.npz                   15.5 MB
   tfidf_vectorizer.pkl                2.1 MB
   title_to_idx.pkl                    2.7 MB


## 18. Dynamic Recommendation Function
Uses **$O(1)$ Lazy Cosine Evaluation**: `Target Vector dot Matrix.T` runs in sub-10 miliseconds rather than trying to construct $87k \times 87k$ matrices (multi-gigabytes of memory out-of-bounds error).

In [30]:
def find_close_match(query, mapping):
    # Perfect substring match fallback logic
    for t in mapping.keys():
        if query in t:
            return mapping[t]
    return None

def recommend_movie(title, top_n=10):
    t0 = time.time()
    
    query = title.lower().strip()
    
    # 1. O(1) Index Match
    idx = title_to_idx.get(query, None)
    if idx is None:
        # Fallback partial matching (for resilience)
        idx = find_close_match(query, title_to_idx)
        if idx is None:
            return f"❌ Movie '{title}' not found."
            
    target_title = movies_content.iloc[idx]["title"]
    
    # 2. Select exactly the single row from the generic TFIDF matrix
    target_vector = tfidf_matrix[idx]
    
    # 3. Dynamic Cosine Similarity against all 87k components 
   # Flatten output directly into a 1D similarity array
    sim_scores = target_vector.dot(tfidf_matrix.T).toarray().flatten()
    
    # 4. Super-Fast Array Partitioning logic
    top_n_plus_one = min(top_n + 1, len(sim_scores))
    top_indices_unsorted = np.argpartition(sim_scores, -top_n_plus_one)[-top_n_plus_one:]
    
    # strictly sort only those tiny subset
    top_indices = top_indices_unsorted[np.argsort(sim_scores[top_indices_unsorted])[::-1]]
    
    # 5. Remove self from the returned scores array subset
    top_indices = top_indices[top_indices != idx][:top_n]
    
    # 6. Structuring output layout
    results = movies_content.iloc[top_indices].copy()
    results["similarity_%"] = (sim_scores[top_indices] * 100).round(1).astype(str) + "%"
    
    res_df = results[["title", "genres", "release_year", "similarity_%"]].reset_index(drop=True)
    
    print(f"⚡ Match on '{target_title}' completed in {(time.time() - t0)*1000:.1f} ms")
    return res_df

# Testing
display(recommend_movie("toy story (1995)", top_n=10))

⚡ Match on 'Toy Story (1995)' completed in 24.5 ms


,title,genres,release_year,similarity_%
0,Toy Story 2 (1999),"[Adventure, Animation, Children, Comedy, Fantasy]",1999,84.4%
1,"Bug's Life, A (1998)","[Adventure, Animation, Children, Comedy]",1998,78.1%
2,Finding Nemo (2003),"[Adventure, Animation, Children, Comedy]",2003,67.4%
3,"Monsters, Inc. (2001)","[Adventure, Animation, Children, Comedy, Fantasy]",2001,65.2%
4,"Incredibles, The (2004)","[Action, Adventure, Animation, Children, Comedy]",2004,64.4%
5,Toy Story 3 (2010),"[Adventure, Animation, Children, Comedy, Fanta...",2010,63.9%
6,Cars (2006),"[Animation, Children, Comedy]",2006,63.5%
7,Finding Dory (2016),"[Adventure, Animation, Comedy]",2016,62.9%
8,Ratatouille (2007),"[Animation, Children, Drama]",2007,58.1%
9,Partly Cloudy (2009),"[Animation, Children, Comedy, Fantasy]",2009,57.8%


## 19. Advanced Pipeline Demo
We can query distinct types of films to qualitatively review NLP clustering accuracy:

In [31]:
print("\n--- Query: 'GoldenEye' ---")
display(recommend_movie("goldeneye"))

print("\n--- Query: 'Pulp Fiction' ---")
display(recommend_movie("pulp fiction"))

print("\n--- Query: 'Matrix, The' ---")
display(recommend_movie("matrix, the"))


--- Query: 'GoldenEye' ---
⚡ Match on 'GoldenEye (1995)' completed in 18.8 ms


,title,genres,release_year,similarity_%
0,"World Is Not Enough, The (1999)","[Action, Adventure, Thriller]",1999,81.7%
1,Die Another Day (2002),"[Action, Adventure, Thriller]",2002,77.9%
2,Dr. No (1962),"[Action, Adventure, Thriller]",1962,76.3%
3,From Russia with Love (1963),"[Action, Adventure, Thriller]",1963,76.2%
4,Tomorrow Never Dies (1997),"[Action, Adventure, Thriller]",1997,73.5%
5,For Your Eyes Only (1981),"[Action, Adventure, Thriller]",1981,67.7%
6,Goldfinger (1964),"[Action, Adventure, Thriller]",1964,67.2%
7,Casino Royale (2006),"[Action, Adventure, Thriller]",2006,67.2%
8,Thunderball (1965),"[Action, Adventure, Thriller]",1965,65.5%
9,Live and Let Die (1973),"[Action, Adventure, Thriller]",1973,64.0%



--- Query: 'Pulp Fiction' ---
⚡ Match on 'Pulp Fiction (1994)' completed in 15.6 ms


,title,genres,release_year,similarity_%
0,Reservoir Dogs (1992),"[Crime, Mystery, Thriller]",1992,73.8%
1,Jackie Brown (1997),"[Crime, Drama, Thriller]",1997,63.9%
2,Kill Bill: Vol. 2 (2004),"[Action, Drama, Thriller]",2004,57.0%
3,Four Rooms (1995),[Comedy],1995,55.9%
4,Kill Bill: Vol. 1 (2003),"[Action, Crime, Thriller]",2003,53.9%
5,Sin City (2005),"[Action, Crime, Film-Noir, Mystery, Thriller]",2005,48.3%
6,Inglourious Basterds (2009),"[Action, Drama, War]",2009,43.8%
7,The Hateful Eight (2015),[Western],2015,43.8%
8,Django Unchained (2012),"[Action, Drama, Western]",2012,43.6%
9,Death Proof (2007),"[Action, Adventure, Crime, Horror, Thriller]",2007,41.1%



--- Query: 'Matrix, The' ---
⚡ Match on 'Matrix, The (1999)' completed in 11.4 ms


,title,genres,release_year,similarity_%
0,"Matrix Reloaded, The (2003)","[Action, Adventure, Sci-Fi, Thriller, IMAX]",2003,79.6%
1,"Matrix Revolutions, The (2003)","[Action, Adventure, Sci-Fi, Thriller, IMAX]",2003,69.3%
2,Dark City (1998),"[Adventure, Film-Noir, Sci-Fi, Thriller]",1998,48.5%
3,Tron (1982),"[Action, Adventure, Sci-Fi]",1982,44.2%
4,V for Vendetta (2006),"[Action, Sci-Fi, Thriller, IMAX]",2006,44.0%
5,Johnny Mnemonic (1995),"[Action, Sci-Fi, Thriller]",1995,42.3%
6,Equilibrium (2002),"[Action, Sci-Fi, Thriller]",2002,41.3%
7,Tron: Legacy (2010),"[Action, Adventure, Sci-Fi, IMAX]",2010,41.0%
8,Gattaca (1997),"[Drama, Sci-Fi, Thriller]",1997,40.4%
9,Ghost in the Shell (Kôkaku kidôtai) (1995),"[Animation, Sci-Fi]",1995,39.8%


## 20. Conclusion & Evaluation Metrics

1. **Qualitative Verification**: Eyeballing results above confirms strong intuitive visual alignment (`GoldenEye` matches strictly 007 action spy thrillers; `Pulp Fiction` matches Tarantino and crime comedies).
2. **Intra-List Diversity Metric**: Raw content-based matching often clusters sequential iterations (i.e. returning *Matrix 2, 3, and 4*). A production wrapper function typically enforces sequence-breaking bounds to improve UX feeling.
3. **Comparison Against Collaborative Filtering**: The major weakness of content-based clustering is its lack of *Serendipity*. It cannot recommend an unexpected correlation (e.g. "Rom-Com fans actually love The Godfather"). Modern Netflix/Spotify algorithms strictly merge this content-based technique (Matrix Dot Product) with User Embeddings (ALS Multi-Stage) into a final **Hybrid Ranking System**.